In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, confusion_matrix
import warnings
warnings.simplefilter('ignore')

In [2]:
# Đọc dữ liệu
data = pd.read_csv(r'.\TrainingWiDS2021.csv', index_col=0)
target_col = 'diabetes_mellitus'

X = data.drop(columns=[target_col])
y = data[target_col]

print(f"Dataset shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts(normalize=True)}")

Dataset shape: (130157, 179)
Target distribution:
diabetes_mellitus
0    0.783715
1    0.216285
Name: proportion, dtype: float64


In [3]:
def engineer_features(train, test=None, y_train=None):
    """
    Áp dụng các kỹ thuật feature engineering từ top solution
    """
    is_train_only = test is None
    
    # Loại bỏ các cột không cần thiết
    cols_to_drop = ['readmission_status', 'encounter_id']
    for col in cols_to_drop:
        if col in train.columns:
            train = train.drop(columns=[col])
        if not is_train_only and col in test.columns:
            test = test.drop(columns=[col])
    
    # ========== TECHNIQUE 1: HOSPITAL ID FREQUENCY ENCODING ==========
    if not is_train_only:
        df_combined = pd.concat([train['hospital_id'], test['hospital_id']])
    else:
        df_combined = train['hospital_id']
    
    hospital_counts = df_combined.value_counts().to_dict()
    train['hospital_id_freq'] = np.log1p(train['hospital_id'].map(hospital_counts))
    if not is_train_only:
        test['hospital_id_freq'] = np.log1p(test['hospital_id'].map(hospital_counts))
    
    # ========== TECHNIQUE 2: HANDLE TRAIN-ONLY/TEST-ONLY CATEGORICAL VALUES ==========
    # Định nghĩa các categorical features
    categoricals = ['elective_surgery', 'ethnicity', 'gender', 'icu_id', 'icu_stay_type', 
                   'icu_type', 'apache_2_diagnosis', 'apache_3j_diagnosis', 'hospital_admit_source']
    
    if not is_train_only:
        for col in categoricals:
            if col in train.columns and col in test.columns:
                train_only_vals = set(train[col].unique()) - set(test[col].unique())
                test_only_vals = set(test[col].unique()) - set(train[col].unique())
                
                train.loc[train[col].isin(train_only_vals), col] = np.nan
                test.loc[test[col].isin(test_only_vals), col] = np.nan
    
    # ========== TECHNIQUE 3: CATEGORICAL TARGET ENCODING (FREQUENCY) ==========
    for col in categoricals:
        if col in train.columns:
            if not is_train_only:
                df_combined = pd.concat([train[col], test[col]])
            else:
                df_combined = train[col]
            
            freq_map = df_combined.value_counts().to_dict()
            train[col + '_freq'] = np.log1p(train[col].map(freq_map))
            if not is_train_only:
                test[col + '_freq'] = np.log1p(test[col].map(freq_map))
    
    # ========== TECHNIQUE 4: CATEGORY AGGREGATION NORMALIZATION ==========
    if 'hospital_admit_source' in train.columns:
        replace_map = {'Other ICU': 'ICU', 'ICU to SDU': 'SDU', 'Step-Down Unit (SDU)': 'SDU',
                      'Other Hospital': 'Other', 'Observation': 'Recovery Room', 
                      'Acute Care/Floor': 'Acute Care'}
        train['hospital_admit_source'] = train['hospital_admit_source'].replace(replace_map)
        if not is_train_only:
            test['hospital_admit_source'] = test['hospital_admit_source'].replace(replace_map)
    
    # ========== TECHNIQUE 5: MIN-MAX RANGE FEATURES ==========
    # Tìm các cột có min/max variants
    max_cols = [col for col in train.columns if 'max' in col]
    for max_col in max_cols:
        base = max_col.split('_max')[0]
        min_col = base + '_min'
        
        if min_col in train.columns:
            # Range (max - min)
            train[base + '_range'] = train[max_col] - train[min_col]
            if not is_train_only:
                test[base + '_range'] = test[max_col] - test[min_col]
            
            # Ratio (max / min) - với xử lý chia cho 0
            train[base + '_ratio'] = np.where(train[min_col] != 0, 
                                              train[max_col] / train[min_col], 
                                              np.nan)
            if not is_train_only:
                test[base + '_ratio'] = np.where(test[min_col] != 0, 
                                                 test[max_col] / test[min_col], 
                                                 np.nan)
    
    # ========== TECHNIQUE 6: SPECIAL GLUCOSE FEATURES ==========
    if 'd1_glucose_max' in train.columns and 'd1_glucose_min' in train.columns:
        train['glucose_range'] = train['d1_glucose_max'] - train['d1_glucose_min']
        train['glucose_range_ratio'] = np.where(train['d1_glucose_min'] != 0,
                                                train['d1_glucose_max'] / train['d1_glucose_min'],
                                                np.nan)
        
        if 'glucose_apache' in train.columns:
            train['glucose_max_diff'] = train['d1_glucose_max'] - train['glucose_apache']
            train['glucose_max_ratio'] = np.where(train['glucose_apache'] != 0,
                                                  train['d1_glucose_max'] / train['glucose_apache'],
                                                  np.nan)
        
        if not is_train_only:
            test['glucose_range'] = test['d1_glucose_max'] - test['d1_glucose_min']
            test['glucose_range_ratio'] = np.where(test['d1_glucose_min'] != 0,
                                                    test['d1_glucose_max'] / test['d1_glucose_min'],
                                                    np.nan)
            
            if 'glucose_apache' in test.columns:
                test['glucose_max_diff'] = test['d1_glucose_max'] - test['glucose_apache']
                test['glucose_max_ratio'] = np.where(test['glucose_apache'] != 0,
                                                      test['d1_glucose_max'] / test['glucose_apache'],
                                                      np.nan)
    
    # ========== TECHNIQUE 7: NUMERICAL RATIO FEATURES ==========
    # Tạo ratio features giữa các numerical columns chính
    num_cols_for_ratio = ['bmi', 'age', 'weight', 'd1_glucose_max', 'd1_glucose_min', 
                          'glucose_apache', 'd1_hco3_max', 'd1_hco3_min', 'pre_icu_los_days']
    
    existing_num_cols = [col for col in num_cols_for_ratio if col in train.columns]
    
    # Tạo một số ratio features quan trọng để tránh explosion
    if 'weight' in existing_num_cols and 'age' in existing_num_cols:
        train['weight_age_ratio'] = np.where(train['age'] != 0, train['weight'] / train['age'], np.nan)
        if not is_train_only:
            test['weight_age_ratio'] = np.where(test['age'] != 0, test['weight'] / test['age'], np.nan)
    
    if 'bmi' in existing_num_cols and 'age' in existing_num_cols:
        train['bmi_age_ratio'] = np.where(train['age'] != 0, train['bmi'] / train['age'], np.nan)
        if not is_train_only:
            test['bmi_age_ratio'] = np.where(test['age'] != 0, test['bmi'] / test['age'], np.nan)
    
    # ========== TECHNIQUE 8: CATEGORICAL INTERACTIONS ==========
    # Tạo một số interaction features chính
    interactions_to_create = [
        ('gender', 'ethnicity'),
        ('icu_type', 'icu_stay_type'),
        ('apache_2_diagnosis', 'apache_3j_diagnosis')
    ]
    
    for col1, col2 in interactions_to_create:
        if col1 in train.columns and col2 in train.columns:
            interaction_name = col1 + '__' + col2
            train[interaction_name] = train[col1].astype(str) + '_' + train[col1].astype(str)
            if not is_train_only:
                test[interaction_name] = test[col1].astype(str) + '_' + test[col2].astype(str)
    
    return train, test


# Áp dụng feature engineering lên toàn bộ dataset (chưa split)
X_engineered, _ = engineer_features(X.copy())

print(f"Features sau engineering: {X_engineered.shape[1]}")
print(f"\nNew features created:")
print(X_engineered.columns.difference(X.columns).tolist())

Features sau engineering: 324

New features created:
['apache_2_diagnosis__apache_3j_diagnosis', 'apache_2_diagnosis_freq', 'apache_3j_diagnosis_freq', 'bmi_age_ratio', 'd1_albumin_range', 'd1_albumin_ratio', 'd1_arterial_pco2_range', 'd1_arterial_pco2_ratio', 'd1_arterial_ph_range', 'd1_arterial_ph_ratio', 'd1_arterial_po2_range', 'd1_arterial_po2_ratio', 'd1_bilirubin_range', 'd1_bilirubin_ratio', 'd1_bun_range', 'd1_bun_ratio', 'd1_calcium_range', 'd1_calcium_ratio', 'd1_creatinine_range', 'd1_creatinine_ratio', 'd1_diasbp_invasive_range', 'd1_diasbp_invasive_ratio', 'd1_diasbp_noninvasive_range', 'd1_diasbp_noninvasive_ratio', 'd1_diasbp_range', 'd1_diasbp_ratio', 'd1_glucose_range', 'd1_glucose_ratio', 'd1_hco3_range', 'd1_hco3_ratio', 'd1_heartrate_range', 'd1_heartrate_ratio', 'd1_hemaglobin_range', 'd1_hemaglobin_ratio', 'd1_hematocrit_range', 'd1_hematocrit_ratio', 'd1_inr_range', 'd1_inr_ratio', 'd1_lactate_range', 'd1_lactate_ratio', 'd1_mbp_invasive_range', 'd1_mbp_invasive

In [4]:
# Chia tập Train/Test có phân tầng (stratify)
X_train, X_test, y_train, y_test = train_test_split(X_engineered, y, 
                                                      test_size=0.2, 
                                                      random_state=42, 
                                                      stratify=y)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"\nTrain target distribution:\n{y_train.value_counts(normalize=True)}")
print(f"\nTest target distribution:\n{y_test.value_counts(normalize=True)}")

X_train shape: (104125, 324)
X_test shape: (26032, 324)

Train target distribution:
diabetes_mellitus
0    0.783712
1    0.216288
Name: proportion, dtype: float64

Test target distribution:
diabetes_mellitus
0    0.783728
1    0.216272
Name: proportion, dtype: float64


In [5]:
# Tự động phân loại features
binary_cols = [col for col in X_train.columns if X_train[col].nunique() == 2]
categorical_cols = [col for col in X_train.select_dtypes(include=['object', 'category']).columns 
                    if col not in binary_cols]
numerical_cols = [col for col in X_train.select_dtypes(include=['int64', 'float64']).columns 
                  if col not in binary_cols]

print(f"Binary columns ({len(binary_cols)}): {binary_cols[:5]}...")
print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols[:5]}...")
print(f"Numerical columns ({len(numerical_cols)}): {numerical_cols[:5]}...")

Binary columns (16): ['elective_surgery', 'gender', 'apache_post_operative', 'arf_apache', 'gcs_unable_apache']...
Categorical columns (8): ['ethnicity', 'hospital_admit_source', 'icu_admit_source', 'icu_stay_type', 'icu_type']...
Numerical columns (300): ['hospital_id', 'age', 'bmi', 'height', 'icu_id']...


In [6]:
# Xây dựng pipeline tiền xử lý
# - Binary: Map về số, NaNs tự động thành -1
binary_transformer = OrdinalEncoder(handle_unknown='use_encoded_value', 
                                    unknown_value=-1, 
                                    encoded_missing_value=-1)

# - Categorical: One-Hot, bỏ qua giá trị lạ và NaN
categorical_transformer = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# - Numerical: Không biến đổi, để mô hình tự xử lý NaN
numerical_transformer = 'passthrough'

preprocessor = ColumnTransformer(transformers=[
    ('bin', binary_transformer, binary_cols),
    ('cat', categorical_transformer, categorical_cols),
    ('num', numerical_transformer, numerical_cols)
])

In [7]:
# HistGradientBoosting tự động xử lý missing values, hỗ trợ class_weight
model = HistGradientBoostingClassifier(class_weight='balanced', random_state=42)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', model)
])

print("Training model...")
pipeline.fit(X_train, y_train)
print("Training complete!")

Training model...
Training complete!


In [8]:
# Dự đoán trên test set
y_pred = pipeline.predict(X_test)
y_pred_proba = pipeline.predict_proba(X_test)[:, 1]

print("--- ĐÁNH GIÁ MÔ HÌNH HISTGRADIENTBOOSTING VỚI FEATURE ENGINEERING ---")
print("\n1. Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\n2. Classification Report:")
print(classification_report(y_test, y_pred))

print(f"\n3. ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}")
print(f"4. PR-AUC (Average Precision): {average_precision_score(y_test, y_pred_proba):.4f}")

--- ĐÁNH GIÁ MÔ HÌNH HISTGRADIENTBOOSTING VỚI FEATURE ENGINEERING ---

1. Confusion Matrix:
[[15996  4406]
 [ 1134  4496]]

2. Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.78      0.85     20402
           1       0.51      0.80      0.62      5630

    accuracy                           0.79     26032
   macro avg       0.72      0.79      0.74     26032
weighted avg       0.84      0.79      0.80     26032


3. ROC-AUC Score: 0.8688
4. PR-AUC (Average Precision): 0.6411


In [10]:
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder

# Chuẩn bị dữ liệu cho LGBM
# Clone dữ liệu để không ảnh hưởng tới X_train, X_test
X_train_lgbm = X_train.copy()
X_test_lgbm = X_test.copy()

# Encode categorical variables
categorical_cols_lgbm = X_train_lgbm.select_dtypes(include=['object']).columns

label_encoders = {}
for col in categorical_cols_lgbm:
    le = LabelEncoder()
    X_train_lgbm[col] = le.fit_transform(X_train_lgbm[col].astype(str))
    X_test_lgbm[col] = le.transform(X_test_lgbm[col].astype(str))
    label_encoders[col] = le

# Fill NaN values with -1 (LGBM sẽ xử lý)
X_train_lgbm = X_train_lgbm.fillna(-1)
X_test_lgbm = X_test_lgbm.fillna(-1)

print(f"LGBM Dataset prepared:")
print(f"X_train_lgbm shape: {X_train_lgbm.shape}")
print(f"X_test_lgbm shape: {X_test_lgbm.shape}")

LGBM Dataset prepared:
X_train_lgbm shape: (104125, 324)
X_test_lgbm shape: (26032, 324)


In [11]:
# Train LGBM Ensemble - Bước 1: Adversarial Validation để lọc train samples
print("Step 1: Adversarial Validation to filter train samples...")

# Tạo dataset cho adversarial validation
# Label 1 = test, Label 0 = train
av_X = pd.concat([X_test_lgbm, X_train_lgbm], axis=0).reset_index(drop=True)
av_y = pd.Series([1] * len(X_test_lgbm) + [0] * len(X_train_lgbm))

# Train LGBM classifier để phân biệt train vs test
av_params = {
    'boosting_type': 'gbdt',
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.01,
    'subsample': 1,
    'colsample_bytree': 0.1,
    'reg_alpha': 3,
    'reg_lambda': 1,
    'n_estimators': 300,
    'random_state': 42,
    'verbose': -1,
}

av_model = lgb.LGBMClassifier(**av_params)
av_model.fit(av_X, av_y)

# Lấy xác suất test-like của train samples
av_proba = av_model.predict_proba(X_train_lgbm)[:, 1]

# Lọc train samples (giữ những samples có xác suất train-like cao)
# Threshold = median để lọc ~50% samples
threshold = np.median(av_proba)
filtered_idx = av_proba <= threshold

X_train_filtered = X_train_lgbm[filtered_idx]
y_train_filtered = y_train[filtered_idx]

print(f"Original train size: {len(X_train_lgbm)}")
print(f"Filtered train size: {len(X_train_filtered)} ({len(X_train_filtered)/len(X_train_lgbm)*100:.1f}%)")
print(f"Target distribution in filtered train:\n{y_train_filtered.value_counts(normalize=True)}")

Step 1: Adversarial Validation to filter train samples...
Original train size: 104125
Filtered train size: 52063 (50.0%)
Target distribution in filtered train:
diabetes_mellitus
0    0.796977
1    0.203023
Name: proportion, dtype: float64


In [12]:
# Step 2: Train LGBM Ensemble với 3 seeds trên filtered train set
print("\nStep 2: Training LGBM ensemble (3 seeds) on filtered train set...")

num_seeds = 3
ensemble_predictions = []

lgbm_params = {
    'boosting_type': 'dart',
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.01,
    'subsample': 1,
    'colsample_bytree': 0.1,
    'reg_alpha': 3,
    'reg_lambda': 1,
    'n_estimators': 14000,
    'random_state': 42,
    'verbose': -1,
}

for seed_idx in range(num_seeds):
    print(f"\n  Training seed {seed_idx + 1}/{num_seeds}...")
    
    # Tạo model với seed khác nhau
    lgbm_params['random_state'] = 666 + seed_idx
    
    lgbm_model = lgb.LGBMClassifier(**lgbm_params)
    
    # Train trên filtered train set
    lgbm_model.fit(
        X_train_filtered, y_train_filtered,
        eval_set=[(X_test_lgbm, y_test)],
        callbacks=[lgb.log_evaluation(period=1000), lgb.early_stopping(stopping_rounds=100)]
    )
    
    # Predict trên test set
    y_pred_proba = lgbm_model.predict_proba(X_test_lgbm)[:, 1]
    ensemble_predictions.append(y_pred_proba)
    
    # Evaluate
    auc_score = roc_auc_score(y_test, y_pred_proba)
    ap_score = average_precision_score(y_test, y_pred_proba)
    print(f"  Seed {seed_idx + 1} - ROC-AUC: {auc_score:.4f}, PR-AUC: {ap_score:.4f}")

print("\n✓ Ensemble training complete!")


Step 2: Training LGBM ensemble (3 seeds) on filtered train set...

  Training seed 1/3...
[1000]	valid_0's auc: 0.84428
[2000]	valid_0's auc: 0.855521
[3000]	valid_0's auc: 0.861479
[4000]	valid_0's auc: 0.86509
[5000]	valid_0's auc: 0.866981
[6000]	valid_0's auc: 0.86814
[7000]	valid_0's auc: 0.868993
[8000]	valid_0's auc: 0.869578
[9000]	valid_0's auc: 0.869785
[10000]	valid_0's auc: 0.870013
[11000]	valid_0's auc: 0.87013
[12000]	valid_0's auc: 0.870119
[13000]	valid_0's auc: 0.870073
[14000]	valid_0's auc: 0.870061
  Seed 1 - ROC-AUC: 0.8701, PR-AUC: 0.6466

  Training seed 2/3...
[1000]	valid_0's auc: 0.841943
[2000]	valid_0's auc: 0.854673
[3000]	valid_0's auc: 0.862613
[4000]	valid_0's auc: 0.865766
[5000]	valid_0's auc: 0.867486
[6000]	valid_0's auc: 0.868603
[7000]	valid_0's auc: 0.869271
[8000]	valid_0's auc: 0.869604
[9000]	valid_0's auc: 0.869943
[10000]	valid_0's auc: 0.870082
[11000]	valid_0's auc: 0.870318
[12000]	valid_0's auc: 0.870301
[13000]	valid_0's auc: 0.870276


In [13]:
# Step 3: Merge predictions sử dụng Geometric Mean
print("\nStep 3: Merging predictions using geometric mean...")

# Chuyển predictions thành array
ensemble_preds_array = np.array(ensemble_predictions)  # shape: (3, n_samples)

# Tính geometric mean: (p1 * p2 * p3)^(1/3)
y_pred_lgbm_ensemble = np.power(np.prod(ensemble_preds_array, axis=0), 1/num_seeds)

print(f"Ensemble predictions shape: {y_pred_lgbm_ensemble.shape}")
print(f"Ensemble predictions range: [{y_pred_lgbm_ensemble.min():.4f}, {y_pred_lgbm_ensemble.max():.4f}]")


Step 3: Merging predictions using geometric mean...
Ensemble predictions shape: (26032,)
Ensemble predictions range: [0.0005, 0.9495]


In [14]:
# Step 4: Đánh giá LGBM Ensemble
print("\n" + "="*70)
print("--- ĐÁNH GIÁ LGBM ENSEMBLE (3 SEEDS + GEOMETRIC MEAN) ---")
print("="*70)

# Dự đoán binary
y_pred_lgbm = (y_pred_lgbm_ensemble >= 0.5).astype(int)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_lgbm)
print("\n1. Confusion Matrix:")
print(cm)

# Classification Report
print("\n2. Classification Report:")
print(classification_report(y_test, y_pred_lgbm))

# ROC-AUC và PR-AUC
auc_lgbm = roc_auc_score(y_test, y_pred_lgbm_ensemble)
ap_lgbm = average_precision_score(y_test, y_pred_lgbm_ensemble)

print(f"\n3. ROC-AUC Score: {auc_lgbm:.4f}")
print(f"4. PR-AUC (Average Precision): {ap_lgbm:.4f}")

print("\n" + "="*70)


--- ĐÁNH GIÁ LGBM ENSEMBLE (3 SEEDS + GEOMETRIC MEAN) ---

1. Confusion Matrix:
[[19055  1347]
 [ 2876  2754]]

2. Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.93      0.90     20402
           1       0.67      0.49      0.57      5630

    accuracy                           0.84     26032
   macro avg       0.77      0.71      0.73     26032
weighted avg       0.83      0.84      0.83     26032


3. ROC-AUC Score: 0.8707
4. PR-AUC (Average Precision): 0.6473

